Эта семинарская тетрадка дополняет слайды презентации, посвященные моделям **мешка слов** и **tf-idf**.

In [ ]:
import nltk
from nltk.corpus import stopwords 
import re
from nltk import word_tokenize
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns #установите, если у вас её нет
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
#nltk.download('stopwords')
#nltk.download('punkt_tab')
stopwords_ru = stopwords.words('russian')


In [ ]:
text = 'Мой дядя самых честных правил, Когда не в шутку занемог, Он уважать себя заставил И лучше выдумать не мог. Его пример другим наука; Но, боже мой, какая скука С больным сидеть и день и ночь, Не отходя ни шагу прочь! Какое низкое коварство Полуживого забавлять, Ему подушки поправлять, Печально подносить лекарство, Вздыхать и думать про себя: Когда же чёрт возьмёт тебя?'

Текст, который мы подадим для создания модели текста, лучше предварительно обработать. Например, удалить стоп-слова и лемматизировать. *Добавим фильтр по POS (из морфологии) для существительных/глаголов.*

In [ ]:
text = re.sub('[^а-яА-ЯёЁ -]', '', text.lower())
print(text)

In [ ]:
from pymorphy3 import MorphAnalyzer 
morph = MorphAnalyzer() 

In [ ]:
lemmatized_text = [morph.parse(tok)[0].normal_form for tok in word_tokenize(text)]
print(lemmatized_text)

In [ ]:
text_no_stop = [' '.join([token for token in lemmatized_text if token not in stopwords_ru])]
print(text_no_stop)

Модель мешка слов очень легко создается с помощью библиотеки **sklearn**. Посмотрим на **униграммы**.

In [ ]:
vectorizer = CountVectorizer(ngram_range=(1, 1))
X = vectorizer.fit_transform(text_no_stop)
print(X)

Достанем, собственно, наши униграммы:

In [ ]:
vectorizer.get_feature_names_out()

И **биграммы**:

In [ ]:
vectorizer_bigrams = CountVectorizer(ngram_range=(2, 2))
X = vectorizer_bigrams.fit_transform(text_no_stop)
vectorizer_bigrams.get_feature_names_out()

In [ ]:
X.toarray()[0]

Объединим частоты и сами биграммы:

In [ ]:
text_vector = pd.DataFrame({'words': vectorizer_bigrams.get_feature_names_out(),
                            'vectors': X.toarray()[0]})
text_vector

Здесь единый и короткий текст, где все леммы являются гапаксами. Давайте возьмем корпус чуть больше, при этом это будут разные документы (главы 'Онегина').

In [ ]:
url = 'https://raw.githubusercontent.com/aleksklimow/DH-programming-2026/refs/heads/main/seminars/EugeneOnegin.txt'
response = requests.get(url)
text = response.text
corpus = re.split(r'ГЛАВА \w+\b', text)
print(len(corpus))
clean_texts = []
for text in corpus:
    text = re.sub(r'\n', ' ', text)
    text = re.sub('[^а-яА-ЯёЁ -]', '', text.lower())
    lemmatized_text = [morph.parse(tok)[0].normal_form for tok in word_tokenize(text)]
    text_no_stop = ' '.join([token for token in lemmatized_text if token not in stopwords_ru])
    clean_texts.append(text_no_stop)
print(clean_texts[0][:100]) #посмотрим на первую главу

In [ ]:
X = vectorizer.fit_transform(clean_texts)
len(vectorizer.get_feature_names_out())

Теперь создадим модель **tf-idf**. min_df=2 для фильтра редких слов.

In [ ]:
tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 1), min_df=2)
X = tfidf_vectorizer.fit_transform(clean_texts)
text_vector = pd.DataFrame(columns = tfidf_vectorizer.get_feature_names_out(), data = X.toarray()) 
text_vector

In [ ]:
print(text_vector['сплин'])

In [ ]:
print(text_vector['онегин'])

In [ ]:
print(text_vector['татьяна'])

Извлечем ключевые слова для первой главы. Добавим сортировку и визуализацию.

In [ ]:
lemmas = list(text_vector.columns)
tf_idf = text_vector.loc[0].tolist()
lemmas_tf_idf = list(zip(lemmas, tf_idf))
lemmas_tf_idf

In [ ]:
sorted_chapter_one = sorted(lemmas_tf_idf, key=lambda x: x[1], reverse = True)
sorted_chapter_one[:20]

In [ ]:
top_10_reversed = sorted_chapter_one[:10][::-1]
top_words = [w[0] for w in top_10_reversed[:10]]
scores = [w[1] for w in top_10_reversed[:10]]
plt.barh(top_words, scores)
plt.xlabel('Значение tf-idf')
plt.title('Ключевые слова первой главы')
plt.show()

**Задание 1.** Сравните ключевые слова во второй главе (дружба Онегина и Ленского) и в шестой главе (убийство Ленского). Выведите топ-20 или топ-30 слов. Как слова отражают смену тем?

In [ ]:
#ваш код здесь

Отрисуем для каждой из глав:

In [ ]:
all_top_words = []
all_top_scores = []

for chapter_idx in range(len(clean_texts)): 
    sorted_chapter = sorted(
        zip(tfidf_vectorizer.get_feature_names_out(), text_vector.iloc[chapter_idx]),
        key=lambda x: x[1],
        reverse=True
    )
    
    top_10_reversed = sorted_chapter[:10][::-1] #топ-10
    
    words = [w[0] for w in top_10_reversed]
    scores = [w[1] for w in top_10_reversed]
    
    all_top_words.append(words)
    all_top_scores.append(scores)

fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(16, 9), sharex=True) #плитка 2х4
fig.suptitle('Топ-10 ключевых слов по главам "Евгения Онегина" (tf-idf)', fontsize=16)

for i, ax in enumerate(axes.flat):
    if i >= 8:
        ax.axis('off')
        continue
    
    ax.barh(all_top_words[i], all_top_scores[i], color='tomato')
    ax.set_title(f'Глава {i+1}', fontsize=11)
    ax.set_xlim(0, max(max(scores) for scores in all_top_scores) * 1.1) 
    
    for j, v in enumerate(all_top_scores[i]):
        ax.text(v + 0.005, j, f'{v:.3f}', va='center', fontsize=9)

plt.tight_layout(rect=[0, 0, 1, 0.96])  
plt.show()

Результат, конечно, далек от совершенства. Отчасти потому, что это довольно искусственное разбиение на отдельные тексты (главы), отчасти потому, что мы могли бы добавить больше слоев предобработки. Например, оставив только существительные. Попробуем:

In [ ]:
clean_texts_nouns = []
for text in corpus:
    text = re.sub(r'\n', ' ', text)
    text = re.sub('[^а-яА-ЯёЁ -]', '', text.lower())
    lemmatized_text = [morph.parse(tok)[0].normal_form for tok in word_tokenize(text) if morph.parse(tok)[0].tag.POS == 'NOUN']
    text_no_stop = ' '.join([token for token in lemmatized_text if token not in stopwords_ru])
    clean_texts_nouns.append(text_no_stop)
print(clean_texts_nouns[:10])

In [ ]:
tfidf_vectorizer_nouns = TfidfVectorizer(ngram_range=(1, 1), min_df=2)
X = tfidf_vectorizer_nouns.fit_transform(clean_texts_nouns)
text_vector_nouns = pd.DataFrame(columns = tfidf_vectorizer_nouns.get_feature_names_out(), data = X.toarray()) 
text_vector_nouns

In [ ]:
all_top_words = []
all_top_scores = []

for chapter_idx in range(len(clean_texts_nouns)):
    sorted_chapter = sorted(
        zip(tfidf_vectorizer_nouns.get_feature_names_out(), text_vector_nouns.iloc[chapter_idx]),
        key=lambda x: x[1],
        reverse=True
    )
    
    top_10_reversed = sorted_chapter[:10][::-1] #топ-10
    words = [w[0] for w in top_10_reversed]
    scores = [w[1] for w in top_10_reversed]
    
    all_top_words.append(words)
    all_top_scores.append(scores)

fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(16, 9), sharex=True) #плитка 2х4
fig.suptitle('Топ-10 ключевых слов по главам "Евгения Онегина" (tf-idf)', fontsize=16)

for i, ax in enumerate(axes.flat):
    if i >= 8:
        ax.axis('off')
        continue
    
    ax.barh(all_top_words[i], all_top_scores[i], color='tomato')
    ax.set_title(f'Глава {i+1}', fontsize=11)
    ax.set_xlim(0, max(max(scores) for scores in all_top_scores) * 1.1) 
    
    for j, v in enumerate(all_top_scores[i]):
        ax.text(v + 0.005, j, f'{v:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

**Задание 2.** Измените код так, чтобы вывести глаголы и прилагательные (раздельно). Интерпретируйте: есть ли в топ-10 глаголов и прилагательных какая-то динамика?

In [ ]:
#ваш код здесь

**Задание 3**. Поработайте с указанным корпусом. Предобработайте его (ведь это очень замусоренный корпус), сделайте уместные (на ваш взгляд) фильтрации по частями речи. В нём тексты 4 разных жанров. Постройте tf-idf модель, сравните ключевые слова. Выведите их на график.

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/aleksklimow/DH-programming-2026/refs/heads/main/seminars/texts_for_tfidfing.csv')
texts = df['texts'].tolist()
#ваш код здесь


**Задание 4: Сравнение моделей**
1. Постройте BoW и TF-IDF на  корпусе. Сравните топ-10 слов: Как TF-IDF лучше выделяет уникальные термины?
2. Визуализируйте wordcloud для tf-idf и мешка слов 30 топ-слов.

In [ ]:
#ваш код здесь

## Сравнение текстов 

До этого момента мы работали только с уровнем слов. Но вообще-то мы можем сравнивать и тексты между собой. Или целые главы. Например, глава 2 и глава 6. Для этого у нас есть метрика косинусной близости (метрика, которая показывает степень схожести двух объектов, измеряя косинус угла между их векторами в многомерном пространстве):

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
vector_1 = [X.toarray()[1]]
vector_2 = [X.toarray()[5]]
cosine_similarity(vector_1, vector_2)

Теперь сравним все главы друг с другом:

In [ ]:
similarity_matrix = cosine_similarity(X.toarray())
similarity_matrix = np.round(similarity_matrix, 2)
similarity_matrix

Осталось поместить на график:

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(
    similarity_matrix,
    annot=True,               
    fmt='.2f',                
    cmap='magma',            
    vmin=0, vmax=1,           
    xticklabels=[f"Гл {i+1}" for i in range(8)],
    yticklabels=[f"Гл {i+1}" for i in range(8)],
    linewidths=0.5,
    cbar_kws={'label': 'Косинусное сходство'}
)

plt.title('Матрица косинусного сходства между главами "Евгения Онегина" (косинусное расстояние tf-idf векторов)', fontsize=14)
plt.xlabel('Глава')
plt.ylabel('Глава')
plt.tight_layout()
plt.show()

Такая матрица симметрична, поэтому иногда маскируют вторую половинку матрицы:

In [ ]:
mask = np.triu(np.ones_like(similarity_matrix, dtype=bool))

plt.figure(figsize=(9, 7))
sns.heatmap(
    similarity_matrix,
    mask=mask,                
    annot=True,
    fmt='.2f',
    cmap='magma',             
    vmin=0.2, vmax=0.9,       
    square=True,              
    xticklabels=[f"Гл {i+1}" for i in range(8)],
    yticklabels=[f"Гл {i+1}" for i in range(8)],
    cbar_kws={'label': 'Косинусное сходство'},
    linewidths=0.4
)

plt.title('Сходство глав "Евгения Онегина" (косинусное расстояние tf-idf векторов)', fontsize=15, pad=15)
plt.tight_layout()
plt.show()

И всё же нам бы хотелось вывести наиболее похожие тексты, не выискивая значения на графике. Можно сделать так:


In [ ]:
for i in range(8):
    similarities = similarity_matrix[i]
    similarities[i] = -1
    
    top_indices = np.argsort(similarities)[-3:][::-1] #выводим топ-3
    top_values  = similarities[top_indices]
    
    print(f'Глава {i+1}:')
    for idx,val in zip(top_indices, top_values):
        print(f'Глава {idx+1:2d}  ({val:.4f})')
    print()

**Задание 5.** Выше мы видим сходство векторных моделей на существительных. Будут ли отличия на глаголах?

In [ ]:
#Ваш код здесь

**Задание 6.** Сравнение текстов.

Вернитесь к тому корпусу, который вы обработали в **Задании 3.** Сравните их вектора друг с другом. Создайте матрицу сходств, визуализируйте результат. Какие тексты ближе всего?

In [ ]:
#Ваш код здесь

## Вектора слов и косинусное расстояние

Мы уже смотрели отдельно вектора слов, но не сравнивали их по сходству. У нас, конечно, очень ограниченный корпус и контексты для обучения, поэтому пока это максимально игрушечный пример (с настоящими векторами слов мы познакомимся позже, когда будем говорить о семантике). Вернемся к нашему раннему примеру, только переведём всё в список:

In [ ]:
oneg = text_vector['онегин'].tolist()
eug = text_vector['евгений'].tolist()
print(oneg)
print(eug)

In [ ]:
tat = text_vector['татьяна'].tolist()
print(tat)

Теперь это два списка чисел - векторов, между которыми мы можем посмотреть расстояние:

In [ ]:
cosine_similarity([oneg, eug])

In [ ]:
cosine_similarity([oneg, tat])

In [ ]:
cosine_similarity([eug, tat])

Немного перепишем, чтобы принимало инпут с клавиатуры:

In [ ]:
cosine_similarity([text_vector[input()].tolist(), text_vector[input()].tolist()])

Осталось визуализировать:

In [ ]:
vectors = np.array([oneg, eug, tat])
labels = ['Онегин', 'Евгений', 'Татьяна']

x_idx = 0  
y_idx = 6

x = vectors[:, x_idx]
y = vectors[:, y_idx]

sim_matrix = cosine_similarity(vectors)

plt.figure(figsize=(7, 7))
plt.scatter(x, y, s=120, c=['#1f77b4', '#ff7f0e', '#2ca02c'], edgecolors='black', zorder=3)

for i, label in enumerate(labels): # Подписи точек
    plt.text(x[i]+0.005, y[i]+0.005, label, fontsize=11, fontweight='bold')

for i in range(3): #рисуем стрелки
    plt.arrow(0, 0, x[i], y[i],  
              head_width=0.008, head_length=0.012, 
              fc=['#1f77b4', '#ff7f0e', '#2ca02c'][i], 
              ec='gray', alpha=0.6, length_includes_head=True)

plt.axhline(0, color='gray', lw=0.6, linestyle='--', alpha=0.5)
plt.axvline(0, color='gray', lw=0.6, linestyle='--', alpha=0.5)
plt.grid(True, alpha=0.15)
plt.xlabel(f'Координата {x_idx+1}')
plt.ylabel(f'Координата {y_idx+1}')
plt.title('Направления Евгения, Онегина и Татьяны')
plt.tight_layout()
plt.show()

**Задание 7.** Поиграйтесь с другими словами. Похожи они или нет? 

In [ ]:
#ваш код здесь